# All algorithms — 80% training / 20% testing

This local notebook covers all 35 standalone score variants, voting ensembles and
stacking models listed in `docs/all-algorithms-8020-20261019/MODEL-GUIDE.md`.
It uses the same 2,734 training and 684 test messages for every model. Existing
completed stages are verified and replayed; set `RUN` to a **new directory** to
perform fresh training. The default training was launched through the CLI, and
this notebook replays its saved evidence. CUDA training uses the existing local
mBERT environment and pinned multilingual cased base. No download is performed.

The corpus and earlier test results were already exposed. This is a retrospective
comparison with inherited labels, source concentration and only nine independent
test components. Rankings are descriptive; no deployment model is selected.


In [1]:
import os, sys, subprocess
from pathlib import Path
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p/'research/smishguard/all_algorithms_8020.py').is_file())
sys.path.insert(0, str(ROOT/'research'))
from smishguard import all_algorithms_8020 as a, nb_comparison as n
RUN = ROOT/'research/runs/all-algorithms-8020-20261019/run-v1'
PYTHON = ROOT/'research/data/mbert-20261002/venv/Scripts/python.exe'
def phase(name):
    env = dict(os.environ, PYTHONPATH=str(ROOT/'research'))
    subprocess.run([str(PYTHON), '-u', '-m', 'smishguard.all_algorithms_8020',
                    name, str(RUN)], cwd=ROOT, env=env, check=True)
print('Run:', RUN)


Run: C:\Users\Jassim Trinidad\OneDrive\Desktop\smishguard\research\runs\all-algorithms-8020-20261019\run-v1


In [2]:
if not RUN.exists():
    phase('prepare')
cfg, rows, splits = a.context(RUN)
audit = n.load(RUN/'data-audit.json')
display(pd.DataFrame(audit['partitions']).T)
print('Fixed classical settings:', len(cfg['matrix']))


,records,ham,smishing,scam_fraction,groups,fraction
train,2734.0,2444.0,290.0,0.106072,291.0,0.799883
test,684.0,611.0,73.0,0.106725,9.0,0.200117


Fixed classical settings: 16


In [3]:
if not (RUN/'classical-results.json').exists():
    phase('classical')
print('Completed classical score variants:', len(n.load(RUN/'classical-results.json')))


Completed classical score variants: 21


In [4]:
if not (RUN/'mbert-complete.json').exists():
    phase('mbert')
mbert = n.load(RUN/'mbert-complete.json')
display(pd.DataFrame([dict(fit=s['id'], training_records=s['train_records'],
                          epochs=s['fixed_epoch'], max_length=s['max_length'],
                          seconds=s['seconds']) for s in mbert['summaries']]))


,fit,training_records,epochs,max_length,seconds
0,mbert,2734,2,128,1232.129604
1,mbert-oof-0,2100,2,128,759.309530
2,mbert-oof-1,1243,2,128,549.607137
3,mbert-oof-2,2125,2,128,783.531797


In [5]:
if not (RUN/'freeze.json').exists():
    phase('freeze')
a.verify(RUN)
print('Frozen configurations:', len(n.load(RUN/'entries.json')))


Frozen configurations: 35


In [6]:
if not (RUN/'test-results.json').exists():
    phase('evaluate')
a.verify(RUN)
results = n.load(RUN/'test-results.json')['results']
table = pd.DataFrame([dict(model=e['id'], **{k:e['test'][k] for k in
        ('accuracy','precision','recall','f1','macro_f1','fpr','fnr')}) for e in results])
display(table.sort_values(['f1','recall'], ascending=False).reset_index(drop=True))
print('Highest observed test F1:', table.loc[table.f1 == table.f1.max(), 'model'].tolist())


,model,accuracy,precision,recall,f1,macro_f1,fpr,fnr
0,latest-B-proposed-raw,0.997076,1.000000,0.972603,0.986111,0.992239,0.000000,0.027397
1,latest-NB-alpha-0.01-raw,0.995614,0.960526,1.000000,0.979866,0.988702,0.004910,0.000000
2,vote-legacy-lr+legacy-nb+mbert,0.995614,1.000000,0.958904,0.979021,0.988286,0.000000,0.041096
3,legacy-nb-sigmoid,0.994152,0.960000,0.986301,0.972973,0.984847,0.004910,0.013699
4,vote-legacy-svm+legacy-nb+mbert,0.994152,1.000000,0.945205,0.971831,0.984284,0.000000,0.054795
5,vote-legacy-svm+mbert,0.992690,0.935897,1.000000,0.966887,0.981389,0.008183,0.000000
6,vote-legacy-nb+mbert,0.992690,0.935897,1.000000,0.966887,0.981389,0.008183,0.000000
7,AE-E-raw,0.992690,0.947368,0.986301,0.966443,0.981171,0.006547,0.013699
8,AE-E-sigmoid,0.992690,0.947368,0.986301,0.966443,0.981171,0.006547,0.013699
9,stack-mbert+legacy-svm+legacy-nb,0.992690,1.000000,0.931507,0.964539,0.980232,0.000000,0.068493


Highest observed test F1: ['latest-B-proposed-raw']


In [7]:
for entry in results:
    prediction = n.load(RUN/'test-predictions'/(entry['id']+'.json'))
    assert prediction['indices'] == splits['partitions']['test']
    assert prediction['labels'] == [rows[i]['label'] for i in prediction['indices']]
    assert n.metrics(prediction['labels'], prediction['predictions']) == entry['test']
phase('report')
print('Verified all', len(results), 'saved metric tables and identical test membership.')
print('Aggregate report:', ROOT/'docs/all-algorithms-8020-20261019/RESULTS.md')


Verified all 35 saved metric tables and identical test membership.
Aggregate report: C:\Users\Jassim Trinidad\OneDrive\Desktop\smishguard\docs\all-algorithms-8020-20261019\RESULTS.md


Training OOF metrics are available in the saved run. The stacking meta model is
trained on base OOF rows; its training fit is not presented as held-out CV.
The raw/sigmoid variants share a classifier, and ensembles share fresh component
fits. Text, vocabulary, weights and row-level predictions remain in private run
folders. The full report explains features, thresholds and limitations. This
notebook does not install a model in the Android app.
